In [4]:
from cube_nn import CubeValueResNet
import torch
from cube import Cube
from solvers import SuperSolver
from cube_nn import NNValueFunctionType
from tqdm import tqdm
import time

In [5]:
# Load NN value function
I = 500
net = CubeValueResNet()
net.load_state_dict(torch.load(f'temp_models/resnet2.2/cube_value_resnet_iter_{I}.pth'))

# Move model to CUDA if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = net.to(device)
print(f'Using device: {device}')

# Use standard value function type
net.set_value_function_type(NNValueFunctionType.STANDARD)

Using device: cuda


/tmp/ipykernel_287590/2195543875.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load(f'temp_models/resnet2.2/cube_value_resnet_iter_{I}.pth'))

In [ ]:

solver = SuperSolver(net.as_value_function(), weight=0.25, noise=0.0,
                            max_moves=40, max_queue_size=1000000, t_max=60, max_restarts=2, batch_size=25, seed=42, dataset_moves=5)

Full optimal value dataset not found at datasets/full/full_5.pt, generating it...
Generating cubes with 1 moves...
Added 18 new cubes.
Generating cubes with 2 moves...
Added 243 new cubes.
Generating cubes with 3 moves...
Added 3240 new cubes.
Generating cubes with 4 moves...
Added 43239 new cubes.
Generating cubes with 5 moves...


In [ ]:
# Generate cubes to evaluate the solver on
N = 100
scramble_moves = 50
cubes = [Cube(n_scramble_moves=scramble_moves, scramble_seed=i) for i in range(N)]

In [5]:
solved_cubes = 0
unsolved_cubes = 0
sol_lengths = []
sol_times = []
for cube in tqdm(cubes, desc=f'Evaluating on cubes', leave=False):
    start_time = time.time()
    solution = solver(cube)
    end_time = time.time()
    if solution is not None:
        solved_cubes += 1
        sol_lengths.append(len(solution))
        sol_times.append(end_time - start_time)
    else:
        unsolved_cubes += 1
        print('Unsolved cubes: ', unsolved_cubes)
print(f'Solved: {solved_cubes}/{N}')
print(f'Fraction solved: {solved_cubes / N}')
print(f'Average solution length: {sum(sol_lengths) / len(sol_lengths) if sol_lengths else 0}')
print(f'Average solution time: {sum(sol_times) / len(sol_times) if sol_times else 0}')

Solved: 100/100
Fraction solved: 1.0
Average solution length: 28.07
Average solution time: 6.939491698741913


In [ ]:
import numpy as np

noises = np.array([0.0, 0.1, 0.2])

solved_fractions = np.zeros((len(noises), 1))
avg_solution_lengths = np.zeros((len(noises), 1))
avg_solution_times = np.zeros((len(noises), 1))

for i, noise in enumerate(noises):
    solver = AStarSolver(net.as_value_function(), weight=0.23, noise=0.0,
                        max_moves=40, max_queue_size=1000000, t_max=60, batch_size=25, max_restarts=2, seed=42)
    
    solved_cubes = 0
    sol_lengths = []
    sol_times = []
    for cube in tqdm(cubes, desc=f'Evaluating noise={noise}', leave=False):
        start_time = time.time()
        solution = solver(cube)
        end_time = time.time()
        if solution is not None:
            solved_cubes += 1
            sol_lengths.append(len(solution))
            sol_times.append(end_time - start_time)
    
    solved_fractions[i, 0] = solved_cubes / N
    avg_solution_lengths[i, 0] = sum(sol_lengths) / len(sol_lengths) if sol_lengths else 0
    avg_solution_times[i, 0] = sum(sol_times) / len(sol_times) if sol_times else 0
    print(f'Noise: {noise}')
    print(f'Solved: {solved_cubes}/{N}')
    print(f'Fraction solved: {solved_fractions[i, 0]}')
    print(f'Average solution length: {avg_solution_lengths[i, 0]}')
    print(f'Average solution time: {avg_solution_times[i, 0]}')

In [ ]:
import numpy as np

weights = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
batch_sizes = np.array([1, 5, 10, 20, 50])

solved_fractions = np.zeros((len(weights), len(batch_sizes)))
avg_solution_lengths = np.zeros((len(weights), len(batch_sizes)))
avg_solution_times = np.zeros((len(weights), len(batch_sizes)))

for i, weight in enumerate(weights):
    for j, batch_size in enumerate(batch_sizes):
        solver = AStarSolver(net.as_value_function(), weight=weight, noise=0.0,
                            max_moves=40, max_queue_size=1000000, t_max=60, batch_size=batch_size, seed=42)
        
        solved_cubes = 0
        sol_lengths = []
        sol_times = []
        for cube in tqdm(cubes, desc=f'Evaluating weight={weight}, batch_size={batch_size}', leave=False):
            start_time = time.time()
            solution = solver(cube)
            end_time = time.time()
            if solution is not None:
                solved_cubes += 1
                sol_lengths.append(len(solution))
                sol_times.append(end_time - start_time)
        
        solved_fractions[i, j] = solved_cubes / N
        avg_solution_lengths[i, j] = sum(sol_lengths) / len(sol_lengths) if sol_lengths else 0
        avg_solution_times[i, j] = sum(sol_times) / len(sol_times) if sol_times else 0
        print(f'Weight: {weight}, Batch Size: {batch_size}')
        print(f'Solved: {solved_cubes}/{N}')
        print(f'Fraction solved: {solved_fractions[i, j]}')
        print(f'Average solution length: {avg_solution_lengths[i, j]}')
        print(f'Average solution time: {avg_solution_times[i, j]}')

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Create meshgrid for plotting
W, B = np.meshgrid(weights, batch_sizes)

# Create figure with subplots for the three metrics
fig = plt.figure(figsize=(18, 5))

# Plot 1: Solved Fraction
ax1 = fig.add_subplot(131, projection='3d')
surf1 = ax1.plot_surface(W, B, solved_fractions.T, cmap='viridis', alpha=0.8, edgecolor='none')
ax1.set_xlabel('Weight')
ax1.set_ylabel('Batch Size')
ax1.set_zlabel('Solved Fraction')
ax1.set_title('Solved Fraction vs Weight and Batch Size')
fig.colorbar(surf1, ax=ax1, shrink=0.5)

# Plot 2: Average Solution Length
ax2 = fig.add_subplot(132, projection='3d')
surf2 = ax2.plot_surface(W, B, avg_solution_lengths.T, cmap='plasma', alpha=0.8, edgecolor='none')
ax2.set_xlabel('Weight')
ax2.set_ylabel('Batch Size')
ax2.set_zlabel('Avg Solution Length')
ax2.set_title('Average Solution Length vs Weight and Batch Size')
fig.colorbar(surf2, ax=ax2, shrink=0.5)

# Plot 3: Average Solution Time
ax3 = fig.add_subplot(133, projection='3d')
surf3 = ax3.plot_surface(W, B, avg_solution_times.T, cmap='coolwarm', alpha=0.8, edgecolor='none')
ax3.set_xlabel('Weight')
ax3.set_ylabel('Batch Size')
ax3.set_zlabel('Avg Solution Time (s)')
ax3.set_title('Average Solution Time vs Weight and Batch Size')
fig.colorbar(surf3, ax=ax3, shrink=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Manual results from previous runs (hardcoded based on print statements)
# These are preserved results so you can visualize without re-running expensive grid search

# Results format: solved_fractions[weight_idx, batch_size_idx]
# weights = [0.1, 0.2, 0.3, 0.4, 0.5]
# batch_sizes = [1, 5, 10, 20, 50]

solved_fractions = np.array([
    [0.52, 0.79, 0.93, 0.96, 0.98],  # weight=0.1
    [0.95, 1.00, 1.00, 0.99, 0.99],  # weight=0.2
    [0.92, 0.98, 1.00, 0.99, 0.99],  # weight=0.3
    [0.66, 0.85, 0.87, 0.94, 0.90],  # weight=0.4
    [0.23, 0.47, 0.45, 0.47, 0.47],  # weight=0.5
])

avg_solution_lengths = np.array([
    [38.00, 36.18, 35.56, 34.05, 31.24],  # weight=0.1
    [33.25, 31.38, 30.00, 29.30, 28.29],  # weight=0.2
    [28.18, 26.59, 26.04, 25.26, 24.87],  # weight=0.3
    [24.64, 24.20, 23.71, 23.24, 22.88],  # weight=0.4
    [22.70, 22.32, 22.07, 21.77, 21.43],  # weight=0.5
])

avg_solution_times = np.array([
    [17.30, 12.10, 8.75, 7.60, 7.42],   # weight=0.1
    [11.13, 8.02, 7.82, 7.37, 7.14],    # weight=0.2
    [16.86, 11.73, 11.86, 10.99, 9.64], # weight=0.3
    [26.13, 18.16, 19.11, 19.60, 18.08],# weight=0.4
    [27.41, 30.99, 30.49, 27.91, 26.57],# weight=0.5
])